# End-to-End US PERM Visa Preprocessing Pipeline
**IT2011 AI/ML Group Assignment** | **Group:** 2026-Y02-S1-MLB-B9G1-08 | **Dataset:** US Permanent Visa Applications (374,362 applications, 2011-2016)

**Problem:** predict whether a PERM labor-certification application will be **denied** (binary classification, 6.9% positive class).

### Group Members & Stage Allocations
| Stage | Member | Technique | Notebook |
|---|---|---|---|
| S1 | IT25101547 | Missing & Invalid Data Handling | `notebooks/IT25101547_MissingData.ipynb` |
| S2 | IT25103364 | Categorical Encoding | `notebooks/IT25103364_Encoding.ipynb` |
| S3 | IT25101145 | Outlier Treatment | `notebooks/IT25101145_OutlierRemoval.ipynb` |
| S4 | IT25102357 | Feature Engineering - Creation | `notebooks/IT25102357_FeatureEngineering.ipynb` |
| S5 | IT25103041 | Normalization & Scaling | `notebooks/IT25103041_Scaling.ipynb` |
| S6 | IT25100285 | Feature Selection & Redundancy Filtering | `notebooks/IT25100285_FeatureSelection.ipynb` |

The stage functions below are **identical** to the code in the member notebooks; the integration check at the end of the pipeline proves both routes produce the same handover dataset.

---

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

# --- Project paths: search upward for the folder that contains data/raw ---
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, 'data', 'raw')) and os.path.dirname(ROOT) != ROOT:
    ROOT = os.path.dirname(ROOT)

RAW_CSV = os.path.join(ROOT, 'data', 'raw', 'us_perm_visas.csv')
EXT_DIR = os.path.join(ROOT, 'data', 'external')
OUT_DIR = os.path.join(ROOT, 'results', 'outputs')
PLOT_DIR = os.path.join(ROOT, 'results', 'eda_visualizations')
LOG_DIR = os.path.join(ROOT, 'results', 'logs')
for _d in (OUT_DIR, PLOT_DIR, LOG_DIR):
    os.makedirs(_d, exist_ok=True)

# --- Shared settings (identical in every notebook) ---
TARGET = 'denied'            # 1 = Denied, 0 = Certified / Certified-Expired
SEED = 42
TEST_SIZE = 0.20             # stratified 80/20 split; teammates use 5-fold CV on the train rows
DEV_SAMPLE = None            # e.g. 0.10 for a fast ~35k-row run; None = full dataset
EDA_ONLY_COLS = ['case_status', 'decision_date']                          # kept for EDA, dropped in S6
LEAKAGE_COLS = ['case_id', 'case_received_date', 'application_type', 'processing_days']  # must never reach a model
NON_FEATURE_COLS = EDA_ONLY_COLS + [TARGET, 'split']

REFERENCE_YEAR = 2016                                                     # last year in the dataset
S4_PASSTHROUGH = ['employer_state', 'job_info_work_state', 'agent_firm_name']  # raw columns kept by S2 for S4
SCALE_COLS = ['wage_offer_annual', 'pw_annual', 'wage_gap_annual', 'wage_to_pw_ratio',  # continuous / ordinal
              'employer_num_employees_log', 'employer_filing_count_log', 'employer_yr_estab', 'employer_age',
              'pw_level', 'edu_worker', 'edu_required', 'edu_gap']

# --- Reference tables (data/external) ---
_states = pd.read_csv(os.path.join(EXT_DIR, 'us_states.csv'))
STATE_LOOKUP = {**dict(zip(_states['state_name'], _states['state_code'])),
                **dict(zip(_states['state_code'], _states['state_code']))}
STATE_REGION = dict(zip(_states['state_code'], _states['census_region']))
VALID_SOC = set(pd.read_csv(os.path.join(EXT_DIR, 'soc_major_groups.csv'), dtype=str)['soc_major_group'])
_naics = pd.read_csv(os.path.join(EXT_DIR, 'naics_sectors.csv'), dtype=str)
VALID_NAICS = set(_naics['naics_2digit'])
NAICS_SECTOR_KEY = dict(zip(_naics['naics_2digit'], _naics['sector_key']))

plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
print(f'Project root: {ROOT}')

## Stage 0 - Chunked load & schema integration + group EDA (m0)

In [ ]:
# ==============================================================================
# STAGE 0: Load & integrate (chunked) — shared by S1 and group_pipeline
# ==============================================================================
# Only these 32 of the 154 raw columns are kept: core employer / job / wage / worker attributes
# plus the target, dates and IDs needed for EDA and de-duplication.
RAW_KEEP = [
    'case_status', 'decision_date', 'case_received_date', 'case_no', 'case_number', 'application_type',
    'employer_name', 'employer_state', 'employer_num_employees', 'employer_yr_estab', 'job_info_work_state',
    'pw_soc_code', 'pw_level_9089', 'pw_amount_9089', 'pw_unit_of_pay_9089', 'pw_source_name_9089',
    'wage_offer_from_9089', 'wage_offered_from_9089', 'wage_offer_unit_of_pay_9089', 'wage_offered_unit_of_pay_9089',
    'naics_2007_us_code', 'naics_us_code', 'naics_code',
    'country_of_citizenship', 'country_of_citzenship', 'class_of_admission',
    'foreign_worker_info_education', 'job_info_education', 'job_info_experience',
    'agent_firm_name', 'refile', 'us_economic_sector',
]
# The CSV merges several versions of the DOL form: the same field has different names per version.
COALESCE_MAP = {
    'case_id': ['case_no', 'case_number'],
    'country_of_citizenship': ['country_of_citizenship', 'country_of_citzenship'],
    'wage_offer_from': ['wage_offer_from_9089', 'wage_offered_from_9089'],
    'wage_offer_unit': ['wage_offer_unit_of_pay_9089', 'wage_offered_unit_of_pay_9089'],
    'naics_code': ['naics_2007_us_code', 'naics_us_code', 'naics_code'],
}
NUMERIC_TEXT_COLS = ['wage_offer_from', 'pw_amount_9089', 'employer_num_employees', 'employer_yr_estab']


def stage0_load_and_integrate(raw_csv=RAW_CSV, dev_sample=DEV_SAMPLE, chunksize=100_000, seed=SEED):
    """
    Reads the 298 MB CSV in 100k-row chunks (never all 154 columns at once):
    - profiles the missing % of all 154 raw columns,
    - keeps only RAW_KEEP, coalesces duplicate-named form fields,
    - parses comma-formatted numbers and ISO dates, normalises text case.
    Returns (dataframe, missingness table).
    """
    n_rows, notna, parts = 0, None, []
    for chunk in pd.read_csv(raw_csv, dtype=str, chunksize=chunksize):
        n_rows += len(chunk)
        counts = chunk.notna().sum()
        notna = counts if notna is None else notna + counts
        parts.append(chunk[RAW_KEEP])
    df = pd.concat(parts, ignore_index=True)
    missing = (1 - notna / n_rows).mul(100).rename('missing_pct_raw').to_frame()

    for new_col, old_cols in COALESCE_MAP.items():
        merged = df[old_cols[0]]
        for col in old_cols[1:]:
            merged = merged.combine_first(df[col])
        df = df.drop(columns=old_cols)
        df[new_col] = merged
        missing.loc[new_col, 'missing_pct_coalesced'] = merged.isna().mean() * 100

    for col in NUMERIC_TEXT_COLS:
        df[col] = pd.to_numeric(df[col].str.replace(',', '', regex=False), errors='coerce')
    for col in ['decision_date', 'case_received_date']:
        df[col] = pd.to_datetime(df[col], format='%Y-%m-%d', errors='coerce')
    for col in ['employer_state', 'job_info_work_state', 'country_of_citizenship', 'employer_name', 'agent_firm_name']:
        df[col] = df[col].str.strip().str.upper()

    if dev_sample:
        df = (df.groupby('case_status', group_keys=False)
                .sample(frac=dev_sample, random_state=seed)
                .sort_index())
    print(f'[Stage 0] Raw file: {n_rows:,} rows x {len(notna)} columns -> kept {df.shape[0]:,} rows x {df.shape[1]} columns')
    return df.reset_index(drop=True), missing

In [ ]:
def plot_dataset_overview(df0, plot_dir=PLOT_DIR, show_plot=False):
    """m0 — group EDA on the Stage-0 data: target framing, denial-rate drift by year, processing-time leakage."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plt.subplots_adjust(wspace=0.3)

    status = df0['case_status'].value_counts().reindex(['Certified', 'Certified-Expired', 'Denied', 'Withdrawn'])
    colors = ['#27ae60', '#27ae60', '#e74c3c', '#95a5a6']
    bars = axes[0].bar(['Certified', 'Certified-\nExpired', 'Denied', 'Withdrawn'], status.values, color=colors)
    axes[0].bar_label(bars, labels=[f'{v:,.0f}\n({v / status.sum():.1%})' for v in status.values], padding=3, fontsize=9)
    axes[0].set_title('case_status -> binary target\n(green = approved 0, red = denied 1, grey = dropped)', fontweight='bold')
    axes[0].set_ylabel('Applications')
    axes[0].margins(y=0.2)

    d = df0[df0['case_status'] != 'Withdrawn']
    yearly = d.groupby(d['decision_date'].dt.year)['case_status'].agg(
        rate=lambda s: (s == 'Denied').mean() * 100, n='size')
    axes[1].plot(yearly.index.astype(int), yearly['rate'], marker='o', linewidth=2.5, color='#8e44ad')
    for yr, row in yearly.iterrows():
        axes[1].annotate(f"{row['rate']:.1f}%\n(n={row['n']:,.0f})", (int(yr), row['rate']),
                         textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
    axes[1].set_title('Denial rate by decision year (concept drift)', fontweight='bold')
    axes[1].set_ylabel('Denied (%)')
    axes[1].set_ylim(0, yearly['rate'].max() * 1.4)

    proc = (df0['decision_date'] - df0['case_received_date']).dt.days
    order = ['Certified', 'Certified-Expired', 'Denied', 'Withdrawn']
    data = [proc[(df0['case_status'] == s) & proc.between(0, 3000)] for s in order]
    axes[2].boxplot(data, showfliers=False, patch_artist=True, boxprops=dict(facecolor='#f5b041'))
    axes[2].set_xticks(range(1, 5), ['Certified', 'Certified-\nExpired', 'Denied', 'Withdrawn'])
    axes[2].set_title('LEAKAGE: processing days (decision - received)\nonly known after the decision -> never a feature',
                      fontweight='bold')
    axes[2].set_ylabel('Days')
    for i, s in enumerate(data, start=1):
        axes[2].text(i, s.median(), f' median {s.median():.0f}', va='bottom', fontsize=9, fontweight='bold')

    plt.suptitle(f'm0: US PERM Visa dataset overview ({len(df0):,} applications, 2011-2016)', fontsize=14, fontweight='bold', y=1.03)
    plt.savefig(f'{plot_dir}/m0_dataset_overview.png', dpi=200, bbox_inches='tight')
    plt.show() if show_plot else plt.close()

## Stage functions (one per member)

In [ ]:
# ==============================================================================
# STAGE 1: Missing & Invalid Data Handling (M1)
# ==============================================================================
UNIT_TO_ANNUAL = {'year': 1, 'yr': 1, 'hour': 2080, 'hr': 2080, 'week': 52, 'wk': 52,
                  'bi-weekly': 26, 'bi': 26, 'month': 12, 'mth': 12}
WAGE_MIN, WAGE_MAX = 10_000, 1_000_000        # plausible annual wage range (USD)
YEAR_MIN, YEAR_MAX = 1800, REFERENCE_YEAR     # plausible employer founding years
MAX_EMPLOYEES = 2_500_000                     # larger than the biggest US employer -> invalid
MISSING_DROP_THRESHOLD = 0.60



def annualise_wage(amount, unit):
    """Convert a wage to USD/year. Missing unit -> infer from magnitude (<= 200 looks hourly). Implausible -> NaN."""
    multiplier = unit.str.strip().str.lower().map(UNIT_TO_ANNUAL)
    inferred = pd.Series(np.where(amount <= 200, 2080, 1), index=amount.index)
    annual = amount * multiplier.fillna(inferred)
    return annual.where(annual.between(WAGE_MIN, WAGE_MAX))


def stage1_missing_and_invalid_data(df, raw_missing=None, plot_dir=PLOT_DIR, show_plot=False):
    """
    1. Target: de-duplicate case IDs (keep latest decision), drop Withdrawn, create `denied`.
    2. Leakage: drop case_id, case_received_date, application_type.
    3. Split: stratified 80/20 -> `split` column. Every statistic from here on is fit on train rows only.
    4. Invalid values: annualise wages, normalise states, validate SOC / NAICS codes,
       implausible employer founding year / size -> NaN.
    5. Missing values: drop columns > 60% missing (train), numeric -> train median, categorical -> 'Unknown'.
    """
    d = df.copy()
    n_raw = len(d)
    d = d.sort_values('decision_date', kind='stable').drop_duplicates('case_id', keep='last').sort_index()
    n_dupes = n_raw - len(d)
    d = d[d['case_status'] != 'Withdrawn'].copy()
    d[TARGET] = (d['case_status'] == 'Denied').astype('int8')
    d = d.drop(columns=['case_id', 'case_received_date', 'application_type']).reset_index(drop=True)

    _, test_idx = train_test_split(d.index, test_size=TEST_SIZE, stratify=d[TARGET], random_state=SEED)
    d['split'] = 'train'
    d.loc[test_idx, 'split'] = 'test'
    train = d['split'].eq('train')

    # --- Invalid values ---
    d['wage_offer_annual'] = annualise_wage(d['wage_offer_from'], d['wage_offer_unit'].fillna(d['pw_unit_of_pay_9089']))
    d['pw_annual'] = annualise_wage(d['pw_amount_9089'], d['pw_unit_of_pay_9089'])
    d = d.drop(columns=['wage_offer_from', 'wage_offer_unit', 'pw_amount_9089', 'pw_unit_of_pay_9089'])
    for col in ['employer_state', 'job_info_work_state']:
        d[col] = d[col].map(STATE_LOOKUP)
    soc = d['pw_soc_code'].str.extract(r'^\s*(\d{2})-\d{4}')[0]
    d['soc_major_group'] = soc.where(soc.isin(VALID_SOC))
    naics = d['naics_code'].str.extract(r'^\s*(\d{2})')[0]
    d['naics_2digit'] = naics.where(naics.isin(VALID_NAICS))
    d = d.drop(columns=['pw_soc_code', 'naics_code'])
    d['employer_yr_estab'] = d['employer_yr_estab'].where(d['employer_yr_estab'].between(YEAR_MIN, YEAR_MAX))
    d['employer_num_employees'] = d['employer_num_employees'].where(
        d['employer_num_employees'].between(1, MAX_EMPLOYEES))

    # --- Missing values ---
    features = [c for c in d.columns if c not in NON_FEATURE_COLS]
    miss_train = d.loc[train, features].isna().mean()
    dropped = miss_train[miss_train > MISSING_DROP_THRESHOLD].index.tolist()
    d = d.drop(columns=dropped)
    features = [c for c in features if c not in dropped]
    numeric = [c for c in features if pd.api.types.is_numeric_dtype(d[c])]
    categorical = [c for c in features if c not in numeric]
    medians = d.loc[train, numeric].median()
    missing_before = d[features].isna().mean() * 100
    d[numeric] = d[numeric].fillna(medians).astype('float32')
    d[categorical] = d[categorical].fillna('Unknown')

    d = d[EDA_ONLY_COLS + sorted(features) + [TARGET, 'split']]
    print(f'[Stage 1] removed {n_dupes:,} duplicate case IDs, dropped Withdrawn -> {len(d):,} rows; '
          f'dropped >60%-missing columns {dropped}; imputed medians {medians.round(0).to_dict()}')

    if plot_dir:
        fig, axes = plt.subplots(1, 2, figsize=(17, 6), gridspec_kw={'width_ratios': [1.25, 1]})
        plt.subplots_adjust(wspace=0.25)
        if raw_missing is not None:
            raw = raw_missing['missing_pct_raw'].dropna().sort_values()
            colors = np.where(raw > MISSING_DROP_THRESHOLD * 100, '#e74c3c', '#27ae60')
            axes[0].bar(range(len(raw)), raw.values, color=colors, width=1.0)
            axes[0].axhline(MISSING_DROP_THRESHOLD * 100, color='black', linestyle='--', linewidth=1)
            axes[0].text(2, MISSING_DROP_THRESHOLD * 100 + 2, '60% drop threshold', fontsize=9)
            for level in (36.1, 59.8, 76.3):
                axes[0].axhline(level, color='#7f8c8d', linestyle=':', linewidth=0.8)
                axes[0].text(len(raw) - 1, level + 1, f'{level}% block (form version)', fontsize=8, ha='right', color='#555')
            axes[0].set_title(f'Missing % of all {len(raw)} raw columns (sorted)\n'
                              f'{(raw > 60).sum()} columns > 60% missing (red)', fontweight='bold')
            axes[0].set_xlabel('Raw columns')
            axes[0].set_ylabel('Missing (%)')
            co = raw_missing.dropna(subset=['missing_pct_coalesced'])
            best_single = [raw_missing.loc[COALESCE_MAP[c], 'missing_pct_raw'].min() for c in co.index]
            x = np.arange(len(co))
            b1 = axes[1].bar(x - 0.2, best_single, 0.4, label='Best single raw column', color='#e67e22')
            b2 = axes[1].bar(x + 0.2, co['missing_pct_coalesced'], 0.4, label='After coalescing form versions', color='#2980b9')
            axes[1].bar_label(b1, fmt='%.1f%%', fontsize=8, padding=2)
            axes[1].bar_label(b2, fmt='%.2f%%', fontsize=8, padding=2)
            axes[1].set_xticks(x, co.index, rotation=20)
            axes[1].set_ylabel('Missing (%)')
            axes[1].set_title('Coalescing duplicate-named form fields\n(remaining NaNs imputed: median / "Unknown")',
                              fontweight='bold')
            axes[1].legend()
            axes[1].margins(y=0.15)
        n_over = int((raw_missing['missing_pct_raw'] > MISSING_DROP_THRESHOLD * 100).sum()) if raw_missing is not None else 0
        plt.suptitle(f'm1: Missing & Invalid Data - {n_over} raw columns are > 60% missing, 5 fields are merged',
                     fontsize=14, fontweight='bold', y=1.03)
        plt.savefig(f'{plot_dir}/m1_missingness_before_after.png', dpi=200, bbox_inches='tight')
        plt.show() if show_plot else plt.close()
    return d

In [ ]:
# ==============================================================================
# STAGE 2: Categorical Encoding (M2)
# ==============================================================================
EDU_ORDER = {'High School': 1, "Associate's": 2, "Bachelor's": 3, "Master's": 4, 'Doctorate': 5}  # Other/Unknown -> 0
PW_LEVEL_ORDER = {'Level I': 1, 'Level II': 2, 'Level III': 3, 'Level IV': 4}                     # Unknown -> 0
TOP_K = {'country_of_citizenship': 20, 'class_of_admission': 10}


def _clean_name(value):
    return ''.join(ch if ch.isalnum() else '_' for ch in str(value)).strip('_')


def one_hot(series, categories, prefix, fallback='Other'):
    """One-hot with a FIXED category list (fit on train). Values outside the list go to `fallback`."""
    cats = list(categories) + ([fallback] if fallback not in categories else [])
    values = pd.Categorical(series.where(series.isin(cats), fallback), categories=cats)
    dummies = pd.get_dummies(values, dtype='int8')
    dummies.columns = [f'{prefix}_{_clean_name(c)}' for c in cats]
    dummies.index = series.index
    return dummies


def stage2_categorical_encoding(df, plot_dir=PLOT_DIR, out_dir=OUT_DIR, show_plot=False):
    """
    - Ordinal: pw_level (I-IV -> 1-4), worker / required education (High School -> Doctorate = 1-5); Unknown/Other -> 0.
    - Binary: experience_required (Y -> 1).
    - Grouped one-hot (category lists fit on train): SOC major group, NAICS sector, employer census region,
      prevailing-wage source, citizenship (top 20 + Other), class of admission (top 10 + Other).
    - Frequency: employer_filing_count = number of OTHER train filings by the same employer (71k employers).
    """
    d = df.copy()
    train = d['split'].eq('train')

    d['pw_level'] = d['pw_level_9089'].map(PW_LEVEL_ORDER).fillna(0).astype('int8')
    d['edu_worker'] = d['foreign_worker_info_education'].map(EDU_ORDER).fillna(0).astype('int8')
    d['edu_required'] = d['job_info_education'].map(EDU_ORDER).fillna(0).astype('int8')
    d['experience_required'] = d['job_info_experience'].eq('Y').astype('int8')

    d['employer_region'] = d['employer_state'].map(STATE_REGION).fillna('Unknown')
    d['naics_sector'] = d['naics_2digit'].map(NAICS_SECTOR_KEY).fillna('Unknown')
    specs = {
        'soc_major_group': 'soc', 'naics_sector': 'naics', 'employer_region': 'region',
        'pw_source_name_9089': 'pwsrc', 'country_of_citizenship': 'country', 'class_of_admission': 'coa',
    }
    fitted = {}
    for col, prefix in specs.items():
        counts = d.loc[train, col].value_counts()
        fitted[col] = sorted(counts.head(TOP_K[col]).index) if col in TOP_K else sorted(counts.index)
    # Top-k columns put rare / unseen values in 'Other'; full-list columns send unseen test values to 'Unknown'
    encoded = [one_hot(d[col], fitted[col], prefix, fallback='Other' if col in TOP_K else 'Unknown')
               for col, prefix in specs.items()]

    counts = d.loc[train, 'employer_name'].value_counts()
    filing = d['employer_name'].map(counts).fillna(0)
    d['employer_filing_count'] = (filing - train.astype(int)).astype('int32')   # exclude the row itself on train

    if plot_dir:
        t = d[train]
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        plt.subplots_adjust(wspace=0.3)
        panels = [
            ('pw_level', {0: 'Unknown', 1: 'I', 2: 'II', 3: 'III', 4: 'IV'}, 'Prevailing-wage level (ordinal)'),
            ('edu_worker', {0: 'Other/\nUnknown', 1: 'High\nSchool', 2: 'Assoc.', 3: 'Bach.', 4: 'Master', 5: 'PhD'},
             'Worker education (ordinal)'),
        ]
        for ax, (col, labels, title) in zip(axes[:2], panels):
            rate = t.groupby(col)[TARGET].mean() * 100
            bars = ax.bar([labels[i] for i in rate.index], rate.values, color='#2980b9')
            ax.bar_label(bars, fmt='%.1f%%', fontsize=9, padding=2)
            ax.axhline(t[TARGET].mean() * 100, color='#e74c3c', linestyle='--', label='overall denial rate')
            ax.set_title(f'Denial rate by {title}', fontweight='bold')
            ax.set_ylabel('Denied (%)')
            ax.legend(fontsize=8)
            ax.margins(y=0.15)
        top = t['country_of_citizenship'].value_counts().head(10).index
        rate = t[t['country_of_citizenship'].isin(top)].groupby('country_of_citizenship')[TARGET].mean().reindex(top) * 100
        bars = axes[2].barh(rate.index[::-1], rate.values[::-1], color='#8e44ad')
        axes[2].bar_label(bars, fmt='%.1f%%', fontsize=8, padding=2)
        axes[2].axvline(t[TARGET].mean() * 100, color='#e74c3c', linestyle='--')
        axes[2].set_title('Denial rate - 10 most common citizenships', fontweight='bold')
        axes[2].set_xlabel('Denied (%)')
        plt.suptitle('m2: Denial risk across encoded categorical attributes (train rows)', fontsize=14, fontweight='bold', y=1.03)
        plt.savefig(f'{plot_dir}/m2_denial_rate_by_category.png', dpi=200, bbox_inches='tight')
        plt.show() if show_plot else plt.close()

    if out_dir:
        maps = {'one_hot_categories': fitted, 'pw_level_order': PW_LEVEL_ORDER, 'education_order': EDU_ORDER}
        with open(os.path.join(out_dir, 'encoding_maps.json'), 'w') as f:
            json.dump(maps, f, indent=2)

    drop = list(specs) + ['pw_level_9089', 'foreign_worker_info_education', 'job_info_education',
                          'job_info_experience', 'naics_2digit', 'employer_name']
    d = pd.concat([d.drop(columns=drop)] + encoded, axis=1)
    features = [c for c in d.columns if c not in NON_FEATURE_COLS]
    return d[EDA_ONLY_COLS + features + [TARGET, 'split']]

In [ ]:
# ==============================================================================
# STAGE 3: Outlier Treatment (M3)
# ==============================================================================
WINSOR_COLS = ['wage_offer_annual', 'pw_annual', 'employer_num_employees_log', 'employer_filing_count_log',
               'employer_yr_estab']
WINSOR_LIMITS = (0.01, 0.99)


def stage3_outlier_treatment(df, plot_dir=PLOT_DIR, out_dir=OUT_DIR, show_plot=False):
    """
    - log1p for the extremely right-skewed counts (employer size, employer filing count).
    - Winsorise at the TRAIN 1st / 99th percentiles. No rows are deleted: unusual applications are
      exactly where denials concentrate, so trimming would throw away minority-class signal.
    """
    d = df.copy()
    train = d['split'].eq('train')
    raw = d[['wage_offer_annual', 'pw_annual', 'employer_num_employees']].copy()

    d['employer_num_employees_log'] = np.log1p(d['employer_num_employees']).astype('float32')
    d['employer_filing_count_log'] = np.log1p(d['employer_filing_count']).astype('float32')
    d = d.drop(columns=['employer_num_employees', 'employer_filing_count'])

    caps = []
    for col in WINSOR_COLS:
        lo, hi = d.loc[train, col].quantile(list(WINSOR_LIMITS))
        clipped = ((d[col] < lo) | (d[col] > hi)).mean() * 100
        d[col] = d[col].clip(lo, hi).astype('float32')
        caps.append({'column': col, 'p01_train': lo, 'p99_train': hi, 'pct_rows_capped': round(clipped, 2)})
    caps = pd.DataFrame(caps)
    if out_dir:
        caps.to_csv(os.path.join(out_dir, 'outlier_caps.csv'), index=False)

    if plot_dir:
        fig, axes = plt.subplots(2, 3, figsize=(18, 8))
        plt.subplots_adjust(hspace=0.45, wspace=0.25)
        panels = [('wage_offer_annual', 'wage_offer_annual', 'Offered wage (USD/yr)'),
                  ('pw_annual', 'pw_annual', 'Prevailing wage (USD/yr)'),
                  ('employer_num_employees', 'employer_num_employees_log', 'Employer size')]
        for j, (raw_col, new_col, label) in enumerate(panels):
            axes[0, j].boxplot(raw[raw_col], vert=False, patch_artist=True, boxprops=dict(facecolor='#e74c3c'))
            axes[0, j].set_title(f'Raw {label}\nmax = {raw[raw_col].max():,.0f}', fontweight='bold')
            axes[1, j].boxplot(d[new_col], vert=False, patch_artist=True, boxprops=dict(facecolor='#2ecc71'))
            suffix = 'log1p + ' if new_col.endswith('_log') else ''
            axes[1, j].set_title(f'Treated: {suffix}winsorised P1-P99 (train)', fontweight='bold')
            axes[0, j].set_yticks([])
            axes[1, j].set_yticks([])
            for ax in (axes[0, j], axes[1, j]) if j < 2 else (axes[0, j],):
                ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v / 1000:,.0f}k'))
        plt.suptitle('m3: Outlier treatment - raw vs treated (no rows removed)', fontsize=14, fontweight='bold', y=1.03)
        plt.savefig(f'{plot_dir}/m3_outlier_boxplots.png', dpi=200, bbox_inches='tight')
        plt.show() if show_plot else plt.close()
    print(f'[Stage 3] caps (train percentiles):\n{caps.to_string(index=False)}')
    return d

In [ ]:
# ==============================================================================
# STAGE 4: Feature Engineering - Creation (M4)
# ==============================================================================
NEW_FEATURES = ['wage_to_pw_ratio', 'wage_below_pw', 'wage_gap_annual', 'employer_age', 'has_agent',
                'same_state', 'edu_meets_requirement', 'edu_gap']


def stage4_feature_engineering(df, plot_dir=PLOT_DIR, show_plot=False):
    """
    Domain features (all available when the application is filed):
    - wage_to_pw_ratio / wage_below_pw / wage_gap_annual: PERM requires the offer to be >= the prevailing wage.
    - employer_age: 2016 - founding year.
    - has_agent: application filed through an attorney / agent firm.
    - same_state: worksite in the employer's home state.
    - edu_meets_requirement / edu_gap: worker education vs the job's required education.
    """
    d = df.copy()
    d['wage_to_pw_ratio'] = (d['wage_offer_annual'] / d['pw_annual'].clip(lower=1)).clip(0, 5).astype('float32')
    d['wage_below_pw'] = (d['wage_to_pw_ratio'] < 1).astype('int8')
    d['wage_gap_annual'] = (d['wage_offer_annual'] - d['pw_annual']).astype('float32')
    d['employer_age'] = (REFERENCE_YEAR - d['employer_yr_estab']).clip(lower=0).astype('float32')
    d['has_agent'] = d['agent_firm_name'].ne('Unknown').astype('int8')
    d['same_state'] = (d['employer_state'].eq(d['job_info_work_state']) & d['employer_state'].ne('Unknown')).astype('int8')
    known = (d['edu_worker'] > 0) & (d['edu_required'] > 0)
    d['edu_meets_requirement'] = (known & (d['edu_worker'] >= d['edu_required'])).astype('int8')
    d['edu_gap'] = np.where(known, d['edu_worker'] - d['edu_required'], 0).astype('int8')
    d = d.drop(columns=S4_PASSTHROUGH)

    if plot_dir:
        t = d[d['split'].eq('train')]
        fig, axes = plt.subplots(1, 2, figsize=(17, 5.5))
        plt.subplots_adjust(wspace=0.25)
        bins = [0, 0.95, 0.999, 1.05, 1.15, 1.3, 1.6, 5.01]
        labels = ['<0.95', '0.95-1.0', '1.0-1.05', '1.05-1.15', '1.15-1.3', '1.3-1.6', '>1.6']
        grp = t.groupby(pd.cut(t['wage_to_pw_ratio'], bins=bins, labels=labels, include_lowest=True), observed=False)[TARGET]
        rate, n = grp.mean() * 100, grp.size()
        axes[0].plot(labels, rate.values, marker='o', linewidth=2.5, color='#8e44ad')
        for i, (r, k) in enumerate(zip(rate.values, n.values)):
            axes[0].annotate(f'{r:.1f}%\n(n={k:,})', (i, r), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
        axes[0].set_title('Denial rate vs offered / prevailing wage ratio', fontweight='bold')
        axes[0].set_xlabel('wage_to_pw_ratio')
        axes[0].set_ylabel('Denied (%)')
        axes[0].set_ylim(0, np.nanmax(rate.values) * 1.35)

        flags = ['has_agent', 'same_state', 'edu_meets_requirement', 'wage_below_pw']
        r0 = [t.loc[t[f] == 0, TARGET].mean() * 100 for f in flags]
        r1 = [t.loc[t[f] == 1, TARGET].mean() * 100 for f in flags]
        x = np.arange(len(flags))
        b0 = axes[1].bar(x - 0.2, r0, 0.4, label='flag = 0', color='#95a5a6')
        b1 = axes[1].bar(x + 0.2, r1, 0.4, label='flag = 1', color='#e67e22')
        axes[1].bar_label(b0, fmt='%.1f%%', fontsize=8, padding=2)
        axes[1].bar_label(b1, fmt='%.1f%%', fontsize=8, padding=2)
        axes[1].set_xticks(x, [f.replace('_', '_\n', 1) for f in flags])
        axes[1].set_ylabel('Denied (%)')
        axes[1].set_title('Denial rate for the new binary features', fontweight='bold')
        axes[1].legend()
        axes[1].margins(y=0.15)
        plt.suptitle('m4: Feature engineering - signal of the created features (train rows)', fontsize=14, fontweight='bold', y=1.03)
        plt.savefig(f'{plot_dir}/m4_engineered_feature_signal.png', dpi=200, bbox_inches='tight')
        plt.show() if show_plot else plt.close()
    print(f'[Stage 4] created {len(NEW_FEATURES)} features: {NEW_FEATURES}')
    return d

In [ ]:
# ==============================================================================
# STAGE 5: Normalization & Feature Scaling (M5)
# ==============================================================================


def stage5_feature_scaling(df, plot_dir=PLOT_DIR, out_dir=OUT_DIR, show_plot=False):
    """Z-score standardisation with mean / std computed on TRAIN rows only; binary and one-hot columns stay 0/1."""
    d = df.copy()
    train = d['split'].eq('train')
    mu = d.loc[train, SCALE_COLS].mean()
    sigma = d.loc[train, SCALE_COLS].std().replace(0, 1.0)
    d[SCALE_COLS] = ((d[SCALE_COLS] - mu) / sigma).astype('float32')
    if out_dir:
        pd.DataFrame({'column': SCALE_COLS, 'train_mean': mu.values, 'train_std': sigma.values}).to_csv(
            os.path.join(out_dir, 'scaler_params.csv'), index=False)

    if plot_dir:
        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        plt.subplots_adjust(hspace=0.4, wspace=0.25)
        for i, (col, unit) in enumerate([('wage_offer_annual', 'USD / year'), ('employer_num_employees_log', 'log(1 + employees)')]):
            before, after = df.loc[train, col], d.loc[train, col]
            axes[i, 0].hist(before, bins=40, color='#3498db', edgecolor='black')
            axes[i, 0].set_title(f'Before: {col}\nmean {before.mean():,.2f} | std {before.std():,.2f}', fontweight='bold')
            axes[i, 0].set_xlabel(unit)
            axes[i, 1].hist(after, bins=40, color='#2ecc71', edgecolor='black')
            axes[i, 1].set_title(f'After z-score: {col}\nmean {after.mean():.2f} | std {after.std():.2f}', fontweight='bold')
            axes[i, 1].set_xlabel('z-score')
        plt.suptitle('m5: Standardisation (train mean / std) - shape preserved, scale aligned', fontsize=14, fontweight='bold', y=1.03)
        plt.savefig(f'{plot_dir}/m5_scaling_comparison.png', dpi=200, bbox_inches='tight')
        plt.show() if show_plot else plt.close()
    return d

In [ ]:
# ==============================================================================
# STAGE 6: Feature Selection (IT25100285)
# ==============================================================================
CORR_THRESHOLD = 0.85        # |r| above this = redundant pair
MI_SAMPLE = 60_000           # stratified train subsample for the mutual-information estimate
MI_CUMULATIVE = 0.90         # keep features until 90% of total MI is covered ...
K_BOUNDS = (15, 30)          # ... but at least 15 and at most 30 features

FEATURE_DESCRIPTIONS = {
    'wage_offer_annual': 'Offered wage, annualised USD (winsorised)',
    'pw_annual': 'Prevailing wage, annualised USD (winsorised)',
    'wage_gap_annual': 'Offered minus prevailing wage, USD/yr',
    'wage_to_pw_ratio': 'Offered / prevailing wage, clipped to [0, 5]',
    'wage_below_pw': '1 if offered wage < prevailing wage',
    'employer_num_employees_log': 'log1p(employer employees), winsorised',
    'employer_filing_count_log': 'log1p(other train filings by the same employer), winsorised',
    'employer_yr_estab': 'Employer founding year (winsorised, median-imputed)',
    'employer_age': '2016 - employer founding year',
    'pw_level': 'Prevailing-wage level I-IV = 1-4, Unknown = 0',
    'edu_worker': 'Worker education, High School=1 ... Doctorate=5, Other/Unknown=0',
    'edu_required': 'Required education, High School=1 ... Doctorate=5, Other/Unknown=0',
    'edu_gap': 'edu_worker - edu_required (0 if either unknown)',
    'edu_meets_requirement': '1 if worker education >= required education (both known)',
    'experience_required': '1 if the job requires experience',
    'has_agent': '1 if filed through an attorney / agent firm',
    'same_state': '1 if worksite state = employer state',
    'denied': 'TARGET: 1 = Denied, 0 = Certified / Certified-Expired',
    'split': 'train / test (stratified 80/20, seed 42)',
}
PREFIX_DESCRIPTIONS = {
    'soc_': 'One-hot SOC occupation major group (2-digit code)', 'naics_': 'One-hot NAICS industry sector',
    'region_': 'One-hot employer census region', 'pwsrc_': 'One-hot prevailing-wage source',
    'country_': 'One-hot citizenship (top 20 + Other)', 'coa_': 'One-hot class of admission (top 10 + Other)',
}


def _describe(col):
    if col in FEATURE_DESCRIPTIONS:
        return FEATURE_DESCRIPTIONS[col]
    return next((desc for p, desc in PREFIX_DESCRIPTIONS.items() if col.startswith(p)), '')


def stage6_feature_selection(df_scaled, df_unscaled, plot_dir=PLOT_DIR, out_dir=OUT_DIR, show_plot=False):
    """
    1. Drop EDA-only columns (case_status, decision_date) and assert no leakage column survived.
    2. Rank every feature by mutual information with `denied` (train subsample).
    3. Greedy correlation filter: walk features by MI, drop any with |r| > 0.85 to an already-kept feature.
    4. Keep the top-k remaining features (90% cumulative MI, bounded 15-30).
    Writes handover A (all features, unscaled) and handover B (selected features, scaled).
    Returns (handover_A, handover_B, selection_report).
    """
    a = df_unscaled.drop(columns=EDA_ONLY_COLS)
    s = df_scaled.drop(columns=EDA_ONLY_COLS)
    leaked = [c for c in LEAKAGE_COLS + EDA_ONLY_COLS if c in a.columns or c in s.columns]
    assert not leaked, f'Leakage columns present: {leaked}'
    assert list(a.columns) == list(s.columns) and len(a) == len(s)

    features = [c for c in s.columns if c not in (TARGET, 'split')]
    train = s['split'].eq('train')
    X, y = s.loc[train, features], s.loc[train, TARGET]
    if len(X) > MI_SAMPLE:
        X_mi, _, y_mi, _ = train_test_split(X, y, train_size=MI_SAMPLE, stratify=y, random_state=SEED)
    else:
        X_mi, y_mi = X, y
    discrete = np.array([c not in SCALE_COLS for c in features])
    mi = pd.Series(mutual_info_classif(X_mi, y_mi, discrete_features=discrete, random_state=SEED), index=features)
    mi = mi.sort_values(ascending=False)

    corr = X.corr().abs()
    kept, dropped_for = [], {}
    for col in mi.index:
        partner = next((k for k in kept if corr.loc[col, k] > CORR_THRESHOLD), None)
        if partner is None:
            kept.append(col)
        else:
            dropped_for[col] = partner
    candidates = mi[kept][mi[kept] > 0]
    cumulative = candidates.cumsum() / candidates.sum()
    k = int(np.clip((cumulative < MI_CUMULATIVE).sum() + 1, *K_BOUNDS))
    selected = candidates.index[:k].tolist()

    report = pd.DataFrame({'feature': mi.index, 'mutual_information': mi.values})
    report['rank'] = np.arange(1, len(report) + 1)
    report['dropped_redundant_with'] = report['feature'].map(dropped_for)
    report['selected'] = report['feature'].isin(selected)

    handover_a = a[features + [TARGET, 'split']]
    handover_b = s[selected + [TARGET, 'split']]

    if out_dir:
        handover_a.to_csv(os.path.join(out_dir, 'cleaned_unscaled_allfeatures.csv.gz'), index=False, float_format='%.6g')
        handover_b.to_csv(os.path.join(out_dir, 'final_processed.csv.gz'), index=False, float_format='%.6g')
        report.to_csv(os.path.join(out_dir, 'feature_selection_report.csv'), index=False)
        with open(os.path.join(out_dir, 'selected_features.txt'), 'w') as f:
            f.write('\n'.join(selected) + '\n')
        dictionary = pd.DataFrame({'column': handover_a.columns})
        dictionary['description'] = dictionary['column'].map(_describe)
        dictionary['dtype'] = [str(handover_a[c].dtype) for c in handover_a.columns]
        dictionary['scaled_in_B'] = dictionary['column'].isin(SCALE_COLS)
        dictionary['in_handover_B'] = dictionary['column'].isin(handover_b.columns)
        dictionary.to_csv(os.path.join(out_dir, 'feature_dictionary.csv'), index=False)

    if plot_dir:
        fig, axes = plt.subplots(1, 2, figsize=(20, 9), gridspec_kw={'width_ratios': [1, 1.1]})
        plt.subplots_adjust(wspace=0.35)
        top = report.head(35).iloc[::-1]
        colors = ['#27ae60' if r.selected else ('#e74c3c' if isinstance(r.dropped_redundant_with, str) else '#bdc3c7')
                  for r in top.itertuples()]
        axes[0].barh(top['feature'], top['mutual_information'], color=colors, edgecolor='black', linewidth=0.4)
        axes[0].set_title(f'Mutual information with `denied` (top 35 of {len(report)})\n'
                          f'green = selected ({k}), red = redundant (|r| > {CORR_THRESHOLD}), grey = below cut-off',
                          fontweight='bold')
        axes[0].set_xlabel('Mutual information (nats, train subsample)')
        axes[0].tick_params(axis='y', labelsize=8)
        sel_corr = X[selected].corr()
        im = axes[1].imshow(sel_corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
        axes[1].set_xticks(range(k), selected, rotation=90, fontsize=7)
        axes[1].set_yticks(range(k), selected, fontsize=7)
        axes[1].set_title(f'Correlation between the {k} selected features (max |r| <= {CORR_THRESHOLD})', fontweight='bold')
        fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
        plt.suptitle('m6: Feature selection - mutual information ranking + redundancy filter', fontsize=14, fontweight='bold', y=1.03)
        plt.savefig(f'{plot_dir}/m6_feature_selection.png', dpi=200, bbox_inches='tight')
        plt.show() if show_plot else plt.close()
    print(f'[Stage 6] {len(features)} candidate features -> {len(dropped_for)} redundant dropped -> {k} selected')
    return handover_a, handover_b, report

## Master pipeline execution harness

In [ ]:
def run_pipeline(dev_sample=DEV_SAMPLE, output_dir=OUT_DIR, plot_dir=PLOT_DIR,
                 log_path=os.path.join(LOG_DIR, 'pipeline_run.log')):
    """Runs Stage 0 -> Stage 6, saves every intermediate output, all EDA plots and a run log."""
    lines, start = [], time.time()

    def log(stage, df, note=''):
        msg = f'{time.strftime("%Y-%m-%d %H:%M:%S")} | {stage:<34} | {df.shape[0]:>9,} rows | {df.shape[1]:>4} cols | {note}'
        print(msg)
        lines.append(msg)

    print('=' * 110)
    print(' EXECUTING END-TO-END GROUP PREPROCESSING PIPELINE (US PERM VISAS)')
    print('=' * 110)
    df_0, raw_missing = stage0_load_and_integrate(dev_sample=dev_sample)
    raw_missing.round(2).to_csv(os.path.join(output_dir, 'raw_missingness.csv'))
    plot_dataset_overview(df_0, plot_dir=plot_dir)
    log('Stage 0 load & integrate', df_0, 'm0 plot saved')

    df_1 = stage1_missing_and_invalid_data(df_0, raw_missing, plot_dir=plot_dir)
    df_1.to_parquet(os.path.join(output_dir, 'stage1_missing_handled.parquet'), index=False)
    log('Stage 1 missing & invalid', df_1, f'denied {df_1[TARGET].mean():.2%}')

    df_2 = stage2_categorical_encoding(df_1, plot_dir=plot_dir, out_dir=output_dir)
    df_2.to_parquet(os.path.join(output_dir, 'stage2_encoded.parquet'), index=False)
    log('Stage 2 categorical encoding', df_2)

    df_3 = stage3_outlier_treatment(df_2, plot_dir=plot_dir, out_dir=output_dir)
    df_3.to_parquet(os.path.join(output_dir, 'stage3_outliers_treated.parquet'), index=False)
    log('Stage 3 outlier treatment', df_3)

    df_4 = stage4_feature_engineering(df_3, plot_dir=plot_dir)
    df_4.to_parquet(os.path.join(output_dir, 'stage4_features_created.parquet'), index=False)
    log('Stage 4 feature engineering', df_4)

    df_5 = stage5_feature_scaling(df_4, plot_dir=plot_dir, out_dir=output_dir)
    df_5.to_parquet(os.path.join(output_dir, 'stage5_scaled.parquet'), index=False)
    log('Stage 5 scaling', df_5)

    handover_a, handover_b, report = stage6_feature_selection(df_5, df_4, plot_dir=plot_dir, out_dir=output_dir)
    log('Stage 6 handover A (all, unscaled)', handover_a)
    log('Stage 6 handover B (selected, scaled)', handover_b)

    lines.append(f'Total runtime: {time.time() - start:.0f}s | dev_sample={dev_sample}')
    with open(log_path, 'w') as f:
        f.write('\n'.join(lines) + '\n')
    print('=' * 110)
    print(f' PIPELINE COMPLETED in {time.time() - start:.0f}s -> {output_dir}/final_processed.csv.gz (+ handover A)')
    print('=' * 110)
    return handover_a, handover_b, report

In [ ]:
# Integration check (part 1): keep the handover B produced by the member notebooks before the pipeline overwrites it
B_PATH = os.path.join(OUT_DIR, 'final_processed.csv.gz')
chain_b = pd.read_csv(B_PATH) if os.path.exists(B_PATH) and DEV_SAMPLE is None else None
print('Notebook-chain output found' if chain_b is not None else 'No notebook-chain output to compare (or DEV_SAMPLE set)')

In [ ]:
# Execute the entire pipeline end-to-end
handover_a, handover_b, selection_report = run_pipeline()
handover_b.head()

In [ ]:
# Integration check (part 2): pipeline output == member-notebook output
if chain_b is not None:
    pd.testing.assert_frame_equal(chain_b, pd.read_csv(B_PATH))
    print('Integration check passed: group_pipeline output is identical to the member notebooks output.')

## Summary of Dataset Handoff and Modelling Readiness

- **Rows:** 354,849 applications after de-duplication and removing *Withdrawn* (24,524 denied = 6.91%).
- **Split:** `split` column - stratified 80/20 (283,879 train / 70,970 test), identical denial rate in both.
- **Leakage-safe:** every imputation value, category list, percentile cap, scaler and the feature selection were fit on train rows only; `decision_date`, `case_status`, processing time, IDs and `application_type` are excluded.
- **Final processed dataset (handover B, DEFAULT)** `results/outputs/final_processed.csv.gz` - 25 selected features, standardised. All six models train on this.
- **Optional variant (handover A)** `results/outputs/cleaned_unscaled_allfeatures.csv.gz` - all 107 features, unscaled, for preprocessing varieties.
- **Column meanings:** `results/outputs/feature_dictionary.csv`.

---
## Model Training - Member Workspaces

**Rules for every member (keeps the 6 models comparable):**
1. Use **handover B** for your main model and for the group comparison. Handover A is optional.
2. Train and tune only on `split == 'train'` with **5-fold `StratifiedKFold(shuffle=True, random_state=42)`** (assignment requires k-fold CV).
3. Evaluate on `split == 'test'` **once**, at the end, with the best configuration.
4. Report for the **Denied** class: precision, recall, F1, plus PR-AUC (`average_precision`), ROC-AUC, macro-F1 and the confusion matrix. Report accuracy only next to the **93.1% "always approve" baseline**.
5. Imbalance: use class weights (or `scale_pos_weight`). SMOTE only inside CV via `imblearn.pipeline.Pipeline` - never on the whole dataset.
6. Final Evaluation needs **several varieties** per model ("different pre-processing and hyperparameter tune-ups"): e.g. different hyperparameter grids, class weights vs SMOTE, an extra pipeline step (PCA, polynomial features) and - optionally - all features from handover A vs the selected features in B.
7. Optional ablation: drop the `pwsrc_*` columns to measure the incomplete-application shortcut (see Stage 1).

In [ ]:
from sklearn.model_selection import StratifiedKFold

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


def load_handover(version='B'):
    """A = all 107 features, unscaled | B = 25 selected features, scaled. Returns X_train, y_train, X_test, y_test."""
    name = {'A': 'cleaned_unscaled_allfeatures.csv.gz', 'B': 'final_processed.csv.gz'}[version]
    df = pd.read_csv(os.path.join(OUT_DIR, name))
    train, test = df[df['split'].eq('train')], df[df['split'].eq('test')]
    return (train.drop(columns=[TARGET, 'split']), train[TARGET],
            test.drop(columns=[TARGET, 'split']), test[TARGET])


X_train, y_train, X_test, y_test = load_handover('B')
print(f'Handover B: X_train {X_train.shape}, X_test {X_test.shape}, train denied rate {y_train.mean():.2%}')

### IT25100285 - Logistic Regression
*Dataset: handover **B** (default). Interpretable baseline; coefficients map directly to the Stage 6 features.*

In [ ]:
# IT25100285 - add Logistic Regression training code here.
# Suggested starting point (replace freely):
#   X_train, y_train, X_test, y_test = load_handover('B')
#   model = LogisticRegression(class_weight='balanced', max_iter=2000)
#   search = GridSearchCV(model, {'C': [0.01, 0.1, 1, 10], 'penalty': ['l1', 'l2'], 'solver': ['saga']},
#                         scoring='average_precision', cv=CV, n_jobs=-1)
#   search.fit(X_train, y_train)

### IT25101547 - Decision Tree
*Dataset: handover **B** (default). Scaling does not affect trees. Optional variety: all 107 features from handover A.*

In [ ]:
# IT25101547 - add Decision Tree training code here.
# Suggested starting point (replace freely):
#   X_train, y_train, X_test, y_test = load_handover('B')
#   model = DecisionTreeClassifier(class_weight='balanced', random_state=SEED)
#   search = GridSearchCV(model, {'max_depth': [5, 10, 20, None], 'min_samples_leaf': [1, 20, 100]},
#                         scoring='average_precision', cv=CV, n_jobs=-1)
#   search.fit(X_train, y_train)

### IT25103364 - Random Forest
*Dataset: handover **B** (default). Run the grid search on a stratified subsample (e.g. 60k rows) if it is slow, then refit on all train rows. Optional variety: handover A.*

In [ ]:
# IT25103364 - add Random Forest training code here.
# Suggested starting point (replace freely):
#   X_train, y_train, X_test, y_test = load_handover('B')
#   model = RandomForestClassifier(class_weight='balanced_subsample', n_jobs=-1, random_state=SEED)
#   search = GridSearchCV(model, {'n_estimators': [200, 400], 'max_depth': [10, 20, None], 'min_samples_leaf': [1, 10]},
#                         scoring='average_precision', cv=CV, n_jobs=-1)
#   search.fit(X_train, y_train)

### IT25101145 - XGBoost (gradient boosting)
*Dataset: handover **B** (default). Usually the strongest model on tabular data. On macOS run `brew install libomp` first (not needed on Colab). Optional variety: handover A.*

In [ ]:
# IT25101145 - add XGBoost (gradient boosting) training code here.
# Suggested starting point (replace freely):
#   X_train, y_train, X_test, y_test = load_handover('B')
#   model = XGBClassifier(scale_pos_weight=13.5, eval_metric='aucpr', n_jobs=-1, random_state=SEED)
#   search = GridSearchCV(model, {'max_depth': [4, 6, 8], 'learning_rate': [0.05, 0.1], 'n_estimators': [300, 600]},
#                         scoring='average_precision', cv=CV, n_jobs=-1)
#   search.fit(X_train, y_train)

### IT25102357 - Support Vector Machine
*Dataset: handover **B** (default). Kernel SVC is ~O(n^2): infeasible on 284k rows x 5 folds. Use LinearSVC on all rows, or RBF on a subsample. No predict_proba - use decision_function for PR-AUC / ROC-AUC.*

In [ ]:
import time, numpy as np, pandas as pd
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             average_precision_score, f1_score, precision_recall_curve)

SEED = 42
df = pd.read_csv('results/outputs/final_processed.csv.gz')
feats = [c for c in df.columns if c not in ('denied', 'split')]
tr, te = df[df.split == 'train'], df[df.split == 'test']
Xtr, ytr, Xte, yte = tr[feats], tr.denied, te[feats], te.denied

# 1) tune C with 5-fold CV on train only (scored by PR-AUC, suited to imbalance)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
best = None
for C in [0.01, 0.1, 1.0]:
    m = LinearSVC(C=C, class_weight='balanced', dual=False, max_iter=5000, random_state=SEED)
    t = time.time()
    s = cross_val_score(m, Xtr, ytr, cv=cv, scoring='average_precision', n_jobs=-1).mean()
    print(f'C={C}: CV PR-AUC={s:.4f} ({time.time()-t:.0f}s)')
    if best is None or s > best[1]: best = (C, s)
C = best[0]; print('best C', C)

# 2) final fit + test evaluation
clf = LinearSVC(C=C, class_weight='balanced', dual=False, max_iter=5000, random_state=SEED).fit(Xtr, ytr)
score = clf.decision_function(Xte); pred = (score > 0).astype(int)
print(confusion_matrix(yte, pred))
print(classification_report(yte, pred, target_names=['Certified', 'Denied'], digits=3))
print('ROC-AUC', round(roc_auc_score(yte, score), 4), 'PR-AUC', round(average_precision_score(yte, score), 4))

# 3) optional: F1-optimal threshold chosen on TRAIN scores (not test)
p, r, th = precision_recall_curve(ytr, clf.decision_function(Xtr))
f1 = 2*p*r/(p+r+1e-12); t_best = th[np.argmax(f1[:-1])]
pred2 = (score > t_best).astype(int)
print(f'threshold {t_best:.3f} -> F1 denied {f1_score(yte, pred2):.3f}')
print(confusion_matrix(yte, pred2))

# 4) coefficients
coef = pd.Series(clf.coef_[0], index=feats).sort_values(key=abs, ascending=False)
print(coef.head(10).round(3))
#IT25102357 - add Support Vector Machine training code here.
# Suggested starting point (replace freely):
#   X_train, y_train, X_test, y_test = load_handover('B')
#   model = LinearSVC(class_weight='balanced', dual=False)  # or SVC(kernel='rbf') on <= 30k stratified rows
#   search = GridSearchCV(model, {'C': [0.01, 0.1, 1, 10]},
#                         scoring='average_precision', cv=CV, n_jobs=-1)
#   search.fit(X_train, y_train)

### IT25103041 - Neural Network (MLP)
*Dataset: handover **B** (default). Replaces KNN (too slow on 284k rows, poor with one-hot distances). No class_weight: use SMOTE inside CV or threshold tuning as the imbalance variety.*

In [ ]:
# IT25103041 - add Neural Network (MLP) training code here.
# Suggested starting point (replace freely):
#   X_train, y_train, X_test, y_test = load_handover('B')
#   model = MLPClassifier(early_stopping=True, random_state=SEED)
#   search = GridSearchCV(model, {'hidden_layer_sizes': [(64,), (128, 64)], 'alpha': [1e-4, 1e-3], 'learning_rate_init': [1e-3, 1e-2]},
#                         scoring='average_precision', cv=CV, n_jobs=-1)
#   search.fit(X_train, y_train)

---
## Group Comparison of the 6 Optimum Models
*To be completed after every member has tuned their model: one table with each model's best CV PR-AUC / F1 (Denied) / recall (Denied) and its single test-set result, then a justification of the best performer.*